### [한번 해보기] 사용자 맞춤 뉴스 추천

In [ ]:
# 1. 뉴스 타이틀 크롤링
# 2. Embedding Model 로드 & Embedding 처리
# 3. FAISS index 생성 + 데이터 추가
# 4. 질의(query) text를 입력 -> Embedding 처리
# 5. 추천 뉴스 타이틀 출력

### 1. 뉴스 타이틀 크롤링

In [2]:
from datetime import datetime, timedelta

def date_list(n_days):
    date_list = []
    today = datetime.now()

    for i in range(n_days):
        target_date = today - timedelta(days=i)
        date_list.append(target_date.strftime('%Y%m%d'))
    
    return date_list

In [16]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import pandas as pd
import time

def article_title_crawl(dates):
    service = Service()
    options = Options()
    prefs = {
    "profile.managed_default_content_settings.images": 2,
    "safebrowsing.enabled": True
    }
    options.add_experimental_option('prefs', prefs)

    for date in dates:
        driver = webdriver.Chrome(service=service, options=options)

        url = f'https://news.naver.com/main/ranking/popularDay.naver?date={date}'

        try:
            driver.get(url)
            time.sleep(2)
            title_tags = driver.find_elements(By.CSS_SELECTOR, 'div.list_content>a')
           
            titles = [title_tag.get_attribute('textContent').strip() for title_tag in title_tags]
            article_urls = [title_tag.get_attribute('href') for title_tag in title_tags]

        except Exception as e:
            print(e)
            driver.quit()
            continue
        
        driver.quit()

        article_df = pd.DataFrame({
            'title': titles,
            'url': article_urls
        })
        article_df.to_csv(f'./data/raw/article_title/{date}.csv', index=False, encoding='utf-8')

    return

In [17]:
dates = date_list(10)
article_title_crawl(dates)

### 2. 데이터 로드

In [24]:
import pandas as pd

article_title_df = pd.DataFrame()
dates = date_list(10)

for date in dates:
    df = pd.read_csv(f'./data/raw/article_title/{date}.csv', encoding='utf-8')
    article_title_df = pd.concat([article_title_df, df], axis=0)

article_title_df

,title,url
0,"아버지·아내·아들 즉사한 공습... 모즈타바, 직전에 건물 떠나 생존",https://n.news.naver.com/article/023/000396499...
1,현관문 막은 박스 20개… 대량 주문 고객 잘못? 택배 기사 분풀이?,https://n.news.naver.com/article/023/000396500...
2,"이란, 2000Km 날아가 때린다... 고체연료 탄도미사일 첫 사용",https://n.news.naver.com/article/023/000396499...
3,"최태원 “2030년까지 반도체 부족... 하이닉스, 美증시 상장도 검토”",https://n.news.naver.com/article/023/000396499...
4,"트럼프의 잠 못 이루는 밤… 직접 전화 받는 대통령, ‘특종’이 쏟아진다",https://n.news.naver.com/article/023/000396496...
...,...,...
415,"계절 알람, ‘냉이낙지전’[주말&]",https://n.news.naver.com/article/145/000002332...
416,"수건, 샤워할 때마다 빤다?… 피부과 전문의가 밝힌 ‘교체 주기’",https://n.news.naver.com/article/145/000002332...
417,엠마 스톤 미묘하게 변한 외모…“마고 로비랑 헷갈려”,https://n.news.naver.com/article/145/000002330...
418,영양소보다 ‘화학 물질’이 더 많은 10가지 식품은?,https://n.news.naver.com/article/145/000002332...


### 2. Embedding Model 로드 & Embedding 처리

In [25]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
titles = [title for title in article_title_df['title']]
title_embed = np.array(
    [embed_model.encode(title) for title in titles],
    dtype='float32'
)

title_idx = [str(idx + 1) for idx in range(len(article_title_df))]

### 3. FAISS index 생성 + 데이터 추가

In [27]:
faiss_idx = faiss.IndexFlatL2(title_embed.shape[1])
faiss_idx.add(title_embed)

### 4. 질의(query) text를 입력 -> Embedding 처리

In [28]:
query_text = '이란과 미국의 전쟁 상황'
query_embed = np.array([embed_model.encode(query_text)], dtype='float32')

##### 5. 추천 뉴스 타이틀 출력

In [29]:
top_n = 5
distances, indices = faiss_idx.search(query_embed, top_n)

In [30]:
for i in range(top_n):
    idx = indices[0][i]
    print(f'추천 {i + 1} | "{article_title_df.iloc[idx]['title']}" (유클리드 거리: {distances[0][i]:.2f}) | {article_title_df.iloc[idx]['url']}')

추천 1 | "운명과 예언, 과학적 분석 어렵던 시대의 미래 예측법" (유클리드 거리: 0.17) | https://n.news.naver.com/article/353/0000054747?ntype=RANKING
추천 2 | "중동전쟁과 미국 리더십의 붕괴 위기" (유클리드 거리: 0.19) | https://n.news.naver.com/article/127/0000038995?ntype=RANKING
추천 3 | "중동전쟁과 미국 리더십의 붕괴 위기" (유클리드 거리: 0.19) | https://n.news.naver.com/article/127/0000038995?ntype=RANKING
추천 4 | "태국 푸껫 해변 외국 관광객들의 알몸 노출로 몸살" (유클리드 거리: 0.24) | https://n.news.naver.com/article/056/0012139536?ntype=RANKING
추천 5 | "알뜰주유소의 배신…석유공사 사장 결국 사과" (유클리드 거리: 0.24) | https://n.news.naver.com/article/374/0000497364?ntype=RANKING
